# Extract places from biographies

Claude Code agents read the biography of each benchmark individual (`01_benchmark_creation.ipynb`), blind to Cultura, and extract the place and the years of each life stage: pre-active, active and post-active life. Agent outputs are cached in `data/extraction/`: an individual already extracted is not sent again. The places are associated with Cliopatria polities in `03_cliopatria_association.ipynb`.

In [1]:
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
FOLDER = ROOT / "notebooks" / "notebooks_polity_validation" / "data"
PROMPTS = ROOT / "notebooks" / "notebooks_polity_validation" / "prompts"
EXTRACTION = FOLDER / "extraction"
LOGS = FOLDER / "logs"
MODEL = "claude-opus-5-5"
STAGES = ["pre_active", "active", "post_active"]
EXTRACTION_GROUP = 10
WORKERS = 10
LIMIT = None
for folder in (EXTRACTION, LOGS):
    folder.mkdir(exist_ok=True)

/Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Benchmark individuals

The individuals whose biography could be downloaded (`scripts/5_texts.py`), one text file each in `data/texts/`.

In [2]:
people = pl.DataFrame([{key: person[key] for key in ("qid", "label", "dictionary_name", "biography_url")}
                       for person in json.loads((FOLDER / "benchmark_texts.json").read_text(encoding="utf-8"))])
if LIMIT:
    people = people.head(LIMIT)
assert people["qid"].is_unique().all()
assert all((FOLDER / "texts" / f"{qid}.txt").exists() for qid in people["qid"])
print(f"{len(people)} individuals")
people.head()

547 individuals


qid,label,dictionary_name,biography_url
str,str,str,str
"""Q9333""","""Laozi""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q4604""","""Confucius""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q47739""","""Zhuang Zhou""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q216072""","""Xunzi""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q7212224""","""Luoxia Hong""","""MacTutor History of Mathematic…","""https://mathshistory.st-andrew…"


### Claude Code runner

Each agent is a headless Claude Code session (`claude -p`) started from the repository root, with only the tools it needs; its full output is kept in `data/logs/`.

In [3]:
def claude(prompt, tools, allowed, log):
    """Run one headless Claude Code session and return its final reply."""
    command = ["claude", "-p", "--model", MODEL, "--output-format", "json", *(["--allowedTools", *allowed] if allowed else []), "--tools", *tools]
    result = subprocess.run(command, input=prompt, cwd=ROOT, capture_output=True, text=True, timeout=3600)
    log.write_text(result.stdout + result.stderr, encoding="utf-8")
    return json.loads(result.stdout)["result"]


def groups(qids, size):
    return [qids[start:start + size] for start in range(0, len(qids), size)]


def done(path, qids):
    return path.exists() and set(qids) <= set(json.loads(path.read_text(encoding="utf-8")))


def run_all(task, jobs, label):
    """Run task(number, qids) for every job in parallel; a failed group is reported and left for the next run."""
    def safe(job):
        try:
            task(*job)
        except Exception as error:
            print(f"{label} group {job[0]:02d} failed: {error}")
    with ThreadPoolExecutor(WORKERS) as pool:
        list(tqdm(pool.map(safe, jobs), total=len(jobs), desc=label))

## Extraction prompt

In [4]:
EXTRACTION_PROMPT = (PROMPTS / "biography_life_stage_extraction.md").read_text(encoding="utf-8").split("---", 1)[1]
Markdown(EXTRACTION_PROMPT)



You are extracting structured data from historians' biographies for a validation benchmark (Nature Scientific Data paper, repo /Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database). Work only from the biography texts; do NOT open any database, parquet, TSV or Cultura file (the comparison must be blind). Do not use the web.

Your individuals (QIDs): {QIDS}
Each text is at notebooks/notebooks_polity_validation/data/texts/{QID}.txt (header gives the name, dictionary and URL). Some are very long (Stanford Encyclopedia of Philosophy entries): read the opening and the life/biography section (use grep for place names, "born", "died", "Life", etc.) rather than the whole entry.

The life of each individual is split into three stages around the active life. The active life runs from the first to the last work, office or professional activity the individual is known for (first publication, first artwork, first post, first reign or command... up to the last one).
- pre_active: from birth to the start of the active life. Where the individual was born, grew up, was educated and trained, before the first work or office.
- active: from the start to the end of the active life. Where the individual worked, held offices, created or published their works (several places allowed).
- post_active: from the end of the active life to death. Where the individual lived after the last work or office (retirement, exile, last residence) and where they died. If the individual was active until death, give the place where they died.

For each individual and each stage, give the years of the stage as the biography gives or implies them, and write the most granular place the biography gives (a city, a region...). If the text says nothing usable for a stage, set that stage to null. Do not use outside knowledge to invent places the text does not mention; you may use your knowledge only to give coordinates.

Output format: the format of notebooks/notebooks_polity_validation/prompts/extraction_format_example.json (read it first as the reference), with the stage keys renamed. Per QID: {"dictionary": <dictionary name from the header>, "pre_active": {...}|null, "active": {...}|null, "post_active": {...}|null}. Each stage object has:
- "place": the place(s) in English, separated by "; "
- "written": the place as written in the text (short, original language)
- "written_en": English translation of "written" (identical if the text is English)
- "verbatim": list of word-for-word extracts from the text supporting the answer. They MUST be exact substrings of the text file (copy-paste, same spelling, accents and spacing); keep each under ~300 characters.
- "verbatim_en": same list translated to English, same length and order (identical if English)
- "confidence": "high" | "medium" | "low"
- "reasoning": one short sentence
- "years": [start_year, end_year] of the stage from the dates of the text (negative for BCE); use null for a bound the text does not let you date, even roughly
- "points": list of [place name, longitude, latitude] for each place (decimal degrees, your best estimate of the place's location)

Write the result as one JSON object (QID -> entry) to notebooks/notebooks_polity_validation/data/extraction/agent_{N}.json (ensure_ascii false, indent 2). Then check it with Python (.venv/bin/python): valid JSON, all your QIDs present, keys pre_active / active / post_active, years a list of two integers or nulls, verbatim and verbatim_en same length, and every verbatim string found in its text file (report and fix any that are not). Reply with a short summary: how many stages filled vs null, and any verbatim you could not match.


### Run the extraction agents

Groups of `EXTRACTION_GROUP` individuals, `WORKERS` agents at a time; individuals already extracted by an earlier run are not sent again. Each agent reads, greps and writes files, and may run `.venv/bin/python` to check its output.

In [5]:
def extract(number, qids):
    path = EXTRACTION / f"agent_{number:03d}.json"
    if not (done(path, qids) or set(qids) <= already_extracted):
        prompt = EXTRACTION_PROMPT.replace("{QIDS}", " ".join(qids)).replace("{N}", f"{number:03d}")
        claude(prompt, ["Read", "Grep", "Glob", "Write", "Bash"], ["Read", "Grep", "Glob", "Write", "Bash(.venv/bin/python:*)"], LOGS / f"extraction_{number:03d}.log")


def extracted():
    return {qid for path in EXTRACTION.glob("agent_*.json") for qid in json.loads(path.read_text(encoding="utf-8"))}


already_extracted = extracted()
extraction_jobs = list(enumerate(groups(people["qid"].to_list(), EXTRACTION_GROUP)))
run_all(extract, extraction_jobs, "extraction")
already_extracted = extracted()
missing = [number for number, qids in extraction_jobs if not set(qids) <= already_extracted]
print(f"{len(extraction_jobs) - len(missing)} / {len(extraction_jobs)} groups extracted; missing: {missing}")

extraction: 100%|██████████| 55/55 [06:31<00:00,  7.11s/it]

55 / 55 groups extracted; missing: []


### Extracted places

Individuals extracted, and stages with a place and with years.

In [6]:
qids = set(people["qid"])
extraction = {qid: person for path in sorted(EXTRACTION.glob("agent_*.json"))
              for qid, person in json.loads(path.read_text(encoding="utf-8")).items() if qid in qids}
print(f"{len(extraction)} / {len(people)} individuals extracted")
filled = pl.DataFrame([{"stage": stage, "place": bool(person.get(stage)),
                        "years": bool(person.get(stage) and any(year is not None for year in person[stage].get("years") or []))}
                       for person in extraction.values() for stage in STAGES])
filled.group_by("stage").agg(individuals=pl.len(), with_place=pl.col("place").sum(), with_years=pl.col("years").sum()).sort(pl.col("stage").cast(pl.Enum(STAGES)))

547 / 547 individuals extracted


stage,individuals,with_place,with_years
str,u32,u32,u32
"""pre_active""",547,437,372
"""active""",547,528,503
"""post_active""",547,392,366


In [7]:
pl.DataFrame([{"qid": qid, "stage": stage, "place": person[stage]["place"], "years": person[stage].get("years"), "confidence": person[stage]["confidence"]}
              for qid, person in extraction.items() for stage in STAGES if person.get(stage)]).sample(8, seed=42)

qid,stage,place,years,confidence
str,str,str,list[i64],str
"""Q748766""","""active""","""Lisbon; Newfoundland""","[1500, 1502]","""medium"""
"""Q334806""","""pre_active""","""Italy""","[null, null]","""low"""
"""Q1288203""","""pre_active""","""Baden (Aargau)""","[1807, 1830]","""medium"""
"""Q1329373""","""post_active""","""Santa Margherita, Procida, nea…","[null, 1931]","""high"""
"""Q299200""","""pre_active""","""Tudela; northern Christian Ibe…","[1075, null]","""medium"""
"""Q799090""","""post_active""","""Aberdyfi""","[1944, 1944]","""low"""
"""Q102371""","""pre_active""","""Peuce Island (Danube delta)""","[null, 395]","""high"""
"""Q264903""","""active""","""Hindestan, Sakastan and Turan;…","[262, 302]","""high"""
